### Data Cleaning

In [1]:
# import libraries
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# load the data
raw = pd.read_csv("../data/Stars.csv")
df = raw.copy()          

In [3]:
# check structure of the data
print("Shape:", df.shape)
print(df.dtypes, "\n")
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Shape: (240, 7)
Temperature         int64
L                 float64
R                 float64
A_M               float64
Color                 str
Spectral_Class        str
Type                int64
dtype: object 

Missing values: 0
Duplicate rows: 0


#### Color column fixing

In [ ]:
# identify unique values in color features before any changes
categorical_features = df.select_dtypes(include='object').columns

for col in categorical_features:
    unique_values = df[col].unique()
    print(f"Unique values in '{col}': {unique_values}")

Unique values in 'Color': <StringArray>
[               'Red',         'Blue White',              'White',
    'Yellowish White',         'Blue white', 'Pale yellow orange',
               'Blue',         'Blue-white',            'Whitish',
       'yellow-white',             'Orange',       'White-Yellow',
              'white',          'yellowish',          'Yellowish',
         'Orange-Red',         'Blue-White']
Length: 17, dtype: str
Unique values in 'Spectral_Class': <StringArray>
['M', 'B', 'A', 'F', 'O', 'K', 'G']
Length: 7, dtype: str


C:\Users\Samsung\AppData\Local\Temp\ipykernel_8840\2056940739.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df.select_dtypes(include='object').columns


In [5]:
# Standardize the 'Color' values formatting
df["Color"] = (df["Color"].str.strip().str.lower()
               .str.replace("-", " ", regex=False)
               .str.replace(r"\s+", " ", regex=True))
df["Spectral_Class"] = df["Spectral_Class"].str.strip().str.upper()

print("Color labels:", raw["Color"].nunique(), "->", df["Color"].nunique())  # 17 -> 12

Color labels: 17 -> 12


In [ ]:
# identify unique values in color features after standardizing format
categorical_features = df.select_dtypes(include='object').columns

for col in categorical_features:
    unique_values = df[col].unique()
    print(f"Unique values in '{col}': {unique_values}")

Unique values in 'Color': <StringArray>
[               'red',         'blue white',              'white',
    'yellowish white', 'pale yellow orange',               'blue',
            'whitish',       'yellow white',             'orange',
       'white yellow',          'yellowish',         'orange red']
Length: 12, dtype: str
Unique values in 'Spectral_Class': <StringArray>
['M', 'B', 'A', 'F', 'O', 'K', 'G']
Length: 7, dtype: str


C:\Users\Samsung\AppData\Local\Temp\ipykernel_8840\2056940739.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df.select_dtypes(include='object').columns


In [ ]:
# add a new column 'Color_Group' for readable EDA plots
# while maintaining granularity for 'Color' column for analysis
color_group_map = {
    "blue white": "blue-white",
    "yellowish white": "yellow-white",
    "yellow white": "yellow-white",
    "white yellow": "yellow-white",
    "whitish": "white",
    "yellowish": "yellow",
    "pale yellow orange": "orange",
    "orange red": "orange",
}
df["Color_Group"] = df["Color"].replace(color_group_map)

print(df["Color"].nunique(), "detailed labels;", df["Color_Group"].nunique(), "groups")

12 detailed labels; 7 groups


In [8]:
df

,Temperature,L,R,A_M,Color,Spectral_Class,Type,Color_Group
0,3068,0.002400,0.1700,16.12,red,M,0,red
1,3042,0.000500,0.1542,16.60,red,M,0,red
2,2600,0.000300,0.1020,18.70,red,M,0,red
3,2800,0.000200,0.1600,16.65,red,M,0,red
4,1939,0.000138,0.1030,20.06,red,M,0,red
...,...,...,...,...,...,...,...,...
235,38940,374830.000000,1356.0000,-9.93,blue,O,5,blue
236,30839,834042.000000,1194.0000,-10.63,blue,O,5,blue
237,8829,537493.000000,1423.0000,-10.73,white,A,5,white
238,9235,404940.000000,1112.0000,-11.23,white,A,5,white


### Type, T, L, R, and Spectral_Class columns validity checks

In [9]:
# Validity checks and readable target labels
type_names = {0: "Red Dwarf", 1: "Brown Dwarf", 2: "White Dwarf",
              3: "Main Sequence", 4: "Super Giant", 5: "Hyper Giant"}

assert df[["Temperature", "L", "R"]].gt(0).all().all(), "non-positive physical value"
assert df["Type"].isin(type_names).all(), "unexpected Type code"
assert set(df["Spectral_Class"]) <= set("OBAFGKM"), "unexpected spectral class"

df["Type_Name"] = df["Type"].map(type_names)
print("All validity checks passed")

All validity checks passed


In [13]:
df

,Temperature,L,R,A_M,Color,Spectral_Class,Type,Color_Group,Type_Name,spectral_temp_mismatch
0,3068,0.002400,0.1700,16.12,red,M,0,red,Red Dwarf,False
1,3042,0.000500,0.1542,16.60,red,M,0,red,Red Dwarf,False
2,2600,0.000300,0.1020,18.70,red,M,0,red,Red Dwarf,False
3,2800,0.000200,0.1600,16.65,red,M,0,red,Red Dwarf,False
4,1939,0.000138,0.1030,20.06,red,M,0,red,Red Dwarf,False
...,...,...,...,...,...,...,...,...,...,...
235,38940,374830.000000,1356.0000,-9.93,blue,O,5,blue,Hyper Giant,False
236,30839,834042.000000,1194.0000,-10.63,blue,O,5,blue,Hyper Giant,False
237,8829,537493.000000,1423.0000,-10.73,white,A,5,white,Hyper Giant,False
238,9235,404940.000000,1112.0000,-11.23,white,A,5,white,Hyper Giant,False


### Consistency check, spectral class versus temperature

In [10]:
# Approximate conventional temperature bounds (K) for each spectral class
bounds = {"O": (30000, np.inf), "B": (10000, 30000), "A": (7500, 10000),
          "F": (6000, 7500), "G": (5200, 6000), "K": (3700, 5200), "M": (0, 3700)}

df["spectral_temp_mismatch"] = [
    not (bounds[s][0] <= t < bounds[s][1])
    for s, t in zip(df["Spectral_Class"], df["Temperature"])
]
print("Mismatches:", df["spectral_temp_mismatch"].sum(), "of", len(df))
print(df.groupby("Spectral_Class")["spectral_temp_mismatch"].agg(["sum", "count"]))

Mismatches: 57 of 240
                sum  count
Spectral_Class            
A                 6     19
B                 4     46
F                12     17
G                 1      1
K                 0      6
M                 6    111
O                28     40


### Outlier review

In [ ]:
# Tukey IQR outlier counts, raw vs log₁₀, for the physical columns.
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75]); i = q3 - q1
    return ((s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)).sum()

rows = []
for c in ["Temperature", "L", "R", "A_M"]:
    log_s = np.log10(df[c]) if c != "A_M" else df[c]
    rows.append({"column": c, "outliers (raw)": iqr_outliers(df[c]),
                 "outliers (log scale)": iqr_outliers(log_s)})
pd.DataFrame(rows)

,column,outliers (raw),outliers (log scale)
0,Temperature,11,0
1,L,12,0
2,R,40,0
3,A_M,0,0


### Physics consistency (Stefan-Boltzmann)

In [ ]:
# Stefan-Boltzmann consistency check: L_expected = R²(T/T☉)⁴ in solar units;
# log₁₀ ratio of catalogued L to expected L should be ≈ 0 if the data is self-consistent.   
expected_L = df["R"] ** 2 * (df["Temperature"] / 5772) ** 4     # solar units
log_ratio = np.log10(df["L"] / expected_L)
print(f"log10(L / L_expected): mean={log_ratio.mean():.2f}, "
      f"std={log_ratio.std():.2f}, max |dev|={log_ratio.abs().max():.2f}")

log10(L / L_expected): mean=-0.29, std=1.15, max |dev|=4.30


### Save

In [ ]:
# save the cleaned data
df.to_csv("../data/stars_clean.csv", index=False)
print("Saved", df.shape, "-> data/stars_clean.csv")

Saved (240, 10) -> data/stars_clean.csv
